In [31]:
from jupyter_dash import JupyterDash
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from animalShelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

image_filename = 'my-image.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

logo = html.Img(
    src='data:image/png;base64,{}'.format(encoded_image.decode()),
    style={'height':'100px'}
)

# Expanded rescue types and breed suitability mapping
rescue_types = [
    {'label': 'Water Rescue', 'value': 'Water Rescue'},
    {'label': 'Mountain/Wilderness Rescue', 'value': 'Wilderness Rescue'},
    {'label': 'Disaster Rescue', 'value': 'Disaster Rescue'},
    {'label': 'Tracking', 'value': 'Tracking'},
    {'label': 'Reset', 'value': 'All'}
]

# Mapping of rescue types to suitable breeds (expanded)
rescue_breed_map = {
    'Water Rescue': [
        'Labrador Retriever Mix', 'Golden Retriever', 'Newfoundland', 'Portuguese Water Dog', 'Chesapeake Bay Retriever'
    ],
    'Wilderness Rescue': [
        'German Shepherd', 'Border Collie', 'Australian Shepherd', 'Siberian Husky', 'Alaskan Malamute'
    ],
    'Disaster Rescue': [
        'German Shepherd', 'Labrador Retriever Mix', 'Golden Retriever', 'Doberman Pinscher', 'Boxer'
    ],
    'Tracking': [
        'Bloodhound', 'German Shepherd', 'Beagle', 'Doberman Pinscher', 'Coonhound'
    ],
}

# Animal types for dropdown
animal_types = [
    {'label': 'Dog', 'value': 'Dog'},
    {'label': 'Cat', 'value': 'Cat'},
    {'label': 'Other', 'value': 'Other'},
    {'label': 'All', 'value': 'All'}
]

app.layout = html.Div([
    logo,
    html.Center(html.B(html.H1('CS-340 Dashboard by Mark Turner - 2025'))),
    html.Hr(),
    html.Div([
        html.Label("Rescue Type:"),
        dcc.RadioItems(
            id='filter-type',
            options=rescue_types,
            value='All',
            inline=True,
            style={'marginBottom': '10px'}
        ),
        html.Label("Animal Type"),
        dcc.Dropdown(
            id='filter-animal-type',
            options=animal_types,
            value='All',
            clearable=False,
            style={'width': '300px'}
        ),
    ], style={'margin':'20px', 'backgroundColor': '#f8f9fa', 'padding': '20px', 'borderRadius': '12px', 'boxShadow': '0 2px 8px rgba(0,0,0,0.07)'}),
    html.Hr(),
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        style_table={'overflowX': 'auto', 'backgroundColor': '#fff', 'borderRadius': '12px', 'boxShadow': '0 2px 8px rgba(0,0,0,0.07)'},
        style_cell={'textAlign': 'left', 'padding': '8px', 'fontSize': '14px'},
        style_header={'backgroundColor': '#0074D9', 'color': 'white', 'fontWeight': 'bold'},
    ),
    html.Br(),
    html.Hr(),
    html.Div([
        html.Div(id='graph-id', style={
            'flex': '1',
            'padding': '20px',
            'backgroundColor': '#fff',
            'borderRadius': '12px',
            'boxShadow': '0 2px 8px rgba(0,0,0,0.07)',
            'marginRight': '10px',
            'minWidth': '0'
        }),
        html.Div(id='map-id', style={
            'flex': '1',
            'padding': '20px',
            'backgroundColor': '#fff',
            'borderRadius': '12px',
            'boxShadow': '0 2px 8px rgba(0,0,0,0.07)',
            'marginLeft': '10px',
            'minWidth': '0'
        })
    ], style={
        'display': 'flex',
        'flexDirection': 'row',
        'justifyContent': 'space-between',
        'alignItems': 'stretch',
        'width': '100%',
        'backgroundColor': '#f0f2f5',
        'padding': '20px 0'
    })
], style={'backgroundColor': '#f0f2f5', 'minHeight': '100vh', 'padding': '0 5vw'})

#############################################
# Interaction Between Components / Controller
#############################################

# Update data table based on filters and animal type
@app.callback(
    Output('datatable-id','data'),
    [Input('filter-type', 'value'),
     Input('filter-animal-type', 'value')]
)
def update_dashboard(filter_type, animal_type):
    # Only include dogs no more than 2 years old (104 weeks) for rescue suitability
    query = {}
    
    if animal_type and animal_type != 'All':
        query['animal_type'] = animal_type
    
    if filter_type and filter_type != 'All':
        suitable_breeds = rescue_breed_map.get(filter_type, [])
        query["age_upon_outcome_in_weeks"] = {"$lte": 104}
        query['breed'] = {'$in': suitable_breeds}
    
    filtered = pd.DataFrame.from_records(db.read(query))

    if '_id' in filtered.columns:
        filtered.drop(columns=['_id'], inplace=True)
    
    return filtered.to_dict('records')

# Display the most common dog breeds in the AAC database based on quantity 
default_graph = [dcc.Graph(figure=px.pie(df, names='breed'))]
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    dff = pd.DataFrame(viewData)
    
    if dff.empty or 'breed' not in dff.columns:
        fig = px.pie(df, names='breed', title='Preferred Animals', height=600, width=600)
        fig.update_layout(margin=dict(l=40, r=40, t=60, b=40), legend=dict(orientation="v", x=1, y=0.5))
        
        return [dcc.Graph(figure=fig, style={'height': '650px', 'width': '100%'})]
    
    # Group with the new breed filtering logic
    breed_counts = dff['breed'].value_counts()
    top_breeds = breed_counts.nlargest(15)
    other_count = breed_counts.iloc[15:].sum()
    pie_data = top_breeds.copy()
    
    if other_count > 0:
        pie_data['Other'] = other_count
    
    fig = px.pie(
        names=pie_data.index,
        values=pie_data.values,
        title='Most common Breeds',
        height=600,
        width=600
    )
    
    fig.update_layout(margin=dict(l=40, r=40, t=60, b=40), legend=dict(orientation="v", x=1, y=0.5))
    return [dcc.Graph(figure=fig, style={'height': '650px', 'width': '100%'})]

# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    dff = pd.DataFrame.from_dict(viewData)

    if not index:
        row = 0
    else: 
        row = index[0]
        
    return [
        dl.Map(dl.Map(style={'width': '100%', 'height': '650px'},
                     center=[dff.iloc[row, 13], dff.iloc[row, 14]],
                     zoom=12,
                     children=[
                         dl.TileLayer(id="base_layer_id"),
                         dl.Marker(position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                            children=[
                                dl.Tooltip(dff.iloc[row, 4]),
                                dl.Popup([
                                    html.H1("Animal Name"),
                                    html.P(dff.iloc[row, 9])
                                ])
                            ])
                     ]))
    ]
    


app.run_server(debug=True)


Successfully connected to Mongo DB
Dash app running on http://127.0.0.1:17258/
